# 计时测量：一份 PDF 走 `run_folder_pipeline` 全流程，时间花在哪

沿用 `run_folder.ipynb` 的环境变量（`NB_PDF_DIR` 等，来自项目根 `.env`）与入库设置（`INGEST_MODE` / `LAYOUT_POLICY` / `REQUALIFY` ...），
对文件夹里前 `MAX_PDFS` 份 PDF 串行跑 ingest → requalify → qualify → index → publish → tree（不答题），并回答：

- **[A]** 每份 PDF 六个顶层阶段各占多少秒 / 百分比；**[A2]** 全部子步骤（pdfspine 抽取 / SVG→PNG / 版面 / chart / 表格 / 描述 / embedding / 落盘 / 各 LLM task）按耗时排名；
- **[B]** LLM / VLM 调用按 task 分组：次数、缓存命中、总 / 平均 / 最大耗时、token；
- **[C]** 本次为每份 PDF 写出的文件数与字节数（按扩展名）。

不改任何源码：用 monkeypatch 在运行期给关键函数包一层计时（跑完恢复）。计时用"自身耗时"口径：每个被包的函数只记扣除了被包子调用之后的时间，
所以 [A2] 各行相加 = 文档总耗时，LLM 等待不会重复计入"版面"或"chart"。

结果同时写到 `<项目根>/data/reports/run-folder/time_measure/summary.json` 与 `summary.md`。

## 冷调用与缓存目录

模型缓存在 `<ingestion 根>/<pdf sha256>/processing/model-cache/`（`JsonCompletionClient(cache_dir=...)`），各阶段结果缓存在同一 `processing/` 下。
为了不碰你正式的 `data/ingestion`，本 notebook **不用**正式 ingestion 根，而是固定写到专用的草稿根 `.../time_measure/ingestion/`；
`COLD_START = True` 时只删除这一个目录（执行前打印路径并校验：必须叫 `ingestion`、父目录叫 `time_measure`、位于 `<项目根>/data/` 之内、不是符号链接）。
`COLD_START = False` 则保留上次草稿根里的缓存（用来看"全部命中缓存"时的耗时）。

## 前置条件

同 `run_folder.ipynb`：项目根 `.env` 配好 LLM（`OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`）、embedding（`OPENAI_EMBEDDING_MODEL` 或 `APP_EMBEDDING_*`）、`NB_PDF_DIR`；
用本仓 `.venv` 作 kernel。可命令行跑：`jupyter nbconvert --to notebook --execute notebooks/time_measure.ipynb --output-dir <某个临时目录>`。

In [ ]:
# editable 安装靠 .pth 把 <repo>/src 加进 sys.path；Databricks 上该 .pth 不生效时显式补上（同 run_folder.ipynb）
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if (_src / "ragspine" / "__init__.py").is_file() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print("Python", sys.version.split()[0], "| src =", _src)

In [ ]:
# ───────────── 参数（只改这一格）─────────────
import shutil

from enterprise_pdf_rag.adapters import folder_pipeline
from enterprise_pdf_rag.adapters.ingest_mode import (
    choose_layout_policy,
    choose_unverified_table_structure,
)
from ragspine.common.evidence.configs import ROOT_DIR, get_settings
from ragspine.common.evidence.providers.providers import load_llm_config

settings = get_settings()

# ---- 本 notebook 的参数 ----
PDF_DIR = (
    settings.pdf_source_dir
)  # 默认沿用 NB_PDF_DIR（run_folder.ipynb 同一变量）；也可直接写 Path("...")
MAX_PDFS = 3  # 只测文件夹里按相对路径排序的前 N 份 PDF
COLD_START = True  # True = 清空专用草稿根（含 model-cache），让本次全是冷调用；False = 保留草稿根缓存（命中缓存会让 LLM 耗时失真）

# ---- 以下与 run_folder.ipynb 保持一致 ----
MAX_LIVE_CALLS_PER_PDF = "auto"
MAX_LIVE_CALLS_TOTAL = None
INGEST_MODE = "lite"
BUILD_TREE = None
LAYOUT_POLICY = "auto"
UNVERIFIED_TABLE_STRUCTURE = "auto"
REQUALIFY = True
# 计时必须串行（每份 PDF 一次一个，耗时才不互相干扰）；不要改成 >1
MAX_PARALLEL_DOCUMENTS = 1

EFFECTIVE_LAYOUT, LAYOUT_REASON = choose_layout_policy(
    LAYOUT_POLICY, ingest_mode=INGEST_MODE, onnx_layout_model=settings.onnx_layout_model
)
EFFECTIVE_TABLE_STRUCTURE, TABLE_STRUCTURE_REASON = choose_unverified_table_structure(
    UNVERIFIED_TABLE_STRUCTURE, ingest_mode=INGEST_MODE
)

# ---- 输出位置：与 run_folder.ipynb 同一个报告根，下面的 time_measure/ 子目录 ----
DATA_DIR = (ROOT_DIR / "data").resolve()
REPORT_DIR = ROOT_DIR / "data" / "reports" / "run-folder"
OUT_DIR = (REPORT_DIR / "time_measure").resolve()
WORK_ROOT = (
    OUT_DIR / "ingestion"
)  # 专用 ingestion 根（草稿）；model-cache 在 WORK_ROOT/<sha256>/processing/model-cache

# ───────────── 参数校验（先于任何写入 / 模型调用）─────────────
if PDF_DIR is None:
    raise RuntimeError("未设置 PDF 文件夹：请在项目根 .env 里设 NB_PDF_DIR，或在本格直接给 PDF_DIR")
PDF_DIR = Path(PDF_DIR).expanduser().resolve()
if not PDF_DIR.is_dir():
    raise RuntimeError(f"PDF 文件夹不存在：{PDF_DIR}")
if not isinstance(MAX_PDFS, int) or MAX_PDFS < 1:
    raise RuntimeError(f"MAX_PDFS 必须是 >=1 的整数，当前 {MAX_PDFS!r}")
if MAX_PARALLEL_DOCUMENTS != 1:
    raise RuntimeError("计时 notebook 要求 MAX_PARALLEL_DOCUMENTS = 1（串行）")
_all_pdfs = folder_pipeline.discover_pdfs(PDF_DIR)
if not _all_pdfs:
    raise RuntimeError(f"{PDF_DIR} 下没有 PDF")
SELECTED_PDFS = _all_pdfs[:MAX_PDFS]
try:
    load_llm_config()
except ValueError as exc:
    raise RuntimeError(
        f"LLM 未配置（{exc}）：在项目根 .env 填 OPENAI_API_KEY / OPENAI_BASE_URL / OPENAI_MODEL"
    ) from exc

# 写入目录护栏：只许写 <项目根>/data/ 下，且绝不写 PDF 源目录
for _label, _path in (("输出目录", OUT_DIR), ("草稿 ingestion 根", WORK_ROOT)):
    _target = Path(_path).resolve()
    if not _target.is_relative_to(DATA_DIR):
        raise RuntimeError(f"{_label} {_target} 不在 {DATA_DIR} 之内")
    if _target.is_relative_to(PDF_DIR) or PDF_DIR.is_relative_to(_target):
        raise RuntimeError(f"{_label} {_target} 与 PDF 源目录 {PDF_DIR} 重叠")

print("PDF 源目录 :", PDF_DIR, f"（共 {len(_all_pdfs)} 份，本次测前 {len(SELECTED_PDFS)} 份）")
for _p in SELECTED_PDFS:
    print("  -", _p.relative_to(PDF_DIR).as_posix(), f"{_p.stat().st_size / 1e6:.1f} MB")
print("入库模式   :", INGEST_MODE, "| 版面:", EFFECTIVE_LAYOUT, f"（{LAYOUT_REASON}）")
print("表格结构   :", EFFECTIVE_TABLE_STRUCTURE, f"（{TABLE_STRUCTURE_REASON}）")
print("requalify  :", REQUALIFY, "| build_tree:", "跟随模式" if BUILD_TREE is None else BUILD_TREE)
print("输出目录   :", OUT_DIR)
print("草稿 ingestion 根:", WORK_ROOT)

# ───────────── 冷启动：只删除专用草稿根 ─────────────
if COLD_START:
    if (
        WORK_ROOT.is_symlink()
        or WORK_ROOT.name != "ingestion"
        or WORK_ROOT.parent.name != "time_measure"
    ):
        raise RuntimeError(f"拒绝删除 {WORK_ROOT}：不是预期的专用草稿根")
    if WORK_ROOT.exists():
        print("COLD_START=True，将删除（仅此目录，含其中全部 model-cache）:", WORK_ROOT)
        shutil.rmtree(WORK_ROOT)
    else:
        print("COLD_START=True，草稿根尚不存在，无需删除:", WORK_ROOT)
else:
    print("COLD_START=False：保留草稿根里已有的缓存（命中缓存时 LLM 耗时会偏小）")

In [ ]:
# ───────────── 计时器：自身耗时口径（扣除被包子调用）─────────────
import functools
import importlib
import inspect
import threading
import time
from collections import defaultdict
from contextlib import contextmanager


class Profiler:
    # 每个 span 记 (pdf, 顶层阶段, 标签) -> 自身秒数；嵌套 span 的耗时从父 span 扣除，所以各标签相加 = 文档总耗时
    def __init__(self):
        self._tls = threading.local()
        self._lock = threading.Lock()
        self.pdf = None  # 当前文档（串行，一个全局即可）
        self.stage = "-"  # 当前顶层阶段
        self.excl = defaultdict(float)
        self.calls = defaultdict(int)
        self.stage_incl = defaultdict(float)
        self.doc_incl = {}
        self.llm = []
        self.kinds = defaultdict(lambda: defaultdict(int))

    @contextmanager
    def span(self, label, *, stage=None, doc=False):
        stack = self._tls.__dict__.setdefault("stack", [])
        if stage is not None:
            self.stage = stage
        key_pdf, key_stage = self.pdf, self.stage
        frame = [time.perf_counter(), 0.0]
        stack.append(frame)
        try:
            yield
        finally:
            elapsed = time.perf_counter() - frame[0]
            stack.pop()
            if stack:
                stack[-1][1] += elapsed
            with self._lock:
                key = (key_pdf, key_stage, label)
                self.excl[key] += elapsed - frame[1]
                self.calls[key] += 1
                if stage is not None:
                    self.stage_incl[(key_pdf, stage)] += elapsed
                    self.stage = "-"
                if doc:
                    self.doc_incl[key_pdf] = elapsed


prof = Profiler()
_PATCHED = []  # (owner, name, original) 用于恢复
_SKIPPED = []  # 没找到的目标（版本差异），只提示


def _wrap(fn, label, *, stage=None, doc=False, before=None):
    @functools.wraps(fn)
    def inner(*args, **kwargs):
        if before is not None:
            before(args, kwargs)
        with prof.span(label, stage=stage, doc=doc):
            return fn(*args, **kwargs)

    return inner


def _wrap_llm(fn):
    # 包 JsonCompletionClient.complete_json / complete_text_json：记 task、耗时、是否命中缓存、token（这两个方法是所有入库 LLM 调用的唯一入口）
    @functools.wraps(fn)
    def inner(self_, *args, **kwargs):
        task = kwargs.get("task", "?")
        rec = {
            "pdf": prof.pdf,
            "stage": prof.stage,
            "task": task,
            "ms": 0.0,
            "cache_hit": False,
            "in_tokens": None,
            "out_tokens": None,
            "error": "",
        }
        started = time.perf_counter()
        try:
            with prof.span(f"LLM {task}"):
                result = fn(self_, *args, **kwargs)
            rec.update(
                cache_hit=bool(result.cache_hit),
                in_tokens=result.input_tokens,
                out_tokens=result.output_tokens,
            )
            return result
        except BaseException as exc:
            rec["error"] = type(exc).__name__
            raise
        finally:
            rec["ms"] = (time.perf_counter() - started) * 1000
            with prof._lock:
                prof.llm.append(rec)

    return inner


def _patch(target, label, **options):
    # target = "包.模块:函数" 或 "包.模块:类.方法"；模块级函数还会替换所有 `from x import f` 出来的同名绑定
    module_name, _, qual = target.partition(":")
    module = importlib.import_module(module_name)
    owner, name = module, qual
    if "." in qual:
        cls_name, name = qual.split(".")
        owner = getattr(module, cls_name, None)
    original = None if owner is None else vars(owner).get(name)
    if original is None or not inspect.isfunction(original):
        _SKIPPED.append(target)
        return
    llm = options.pop("llm", False)
    wrapper = _wrap_llm(original) if llm else _wrap(original, label, **options)
    owners = [owner]
    if owner is module:
        owners = [
            mod
            for mod_name, mod in list(sys.modules.items())
            if mod is not None
            and mod_name.startswith(("enterprise_pdf_rag", "ragspine"))
            and vars(mod).get(name) is original
        ]
    for each in owners:
        setattr(each, name, wrapper)
        _PATCHED.append((each, name, original))


def restore_patches():
    while _PATCHED:
        owner, name, original = _PATCHED.pop()
        setattr(owner, name, original)


_A = "enterprise_pdf_rag.adapters."


def _set_pdf(args, kwargs):
    prof.pdf = Path(args[0]).name  # _run_document(pdf, digest, ...)


def _count_kind(args, kwargs):
    prof.kinds[prof.pdf][args[2].kind.value] += 1  # SemanticObjectAdapter.process(self, page, item)


def install_patches():
    # 文档 / 六个顶层阶段（folder_pipeline 里按这些名字调用）
    _patch(_A + "folder_pipeline:_run_document", "(文档其余)", doc=True, before=_set_pdf)
    for target, stage in (
        (_A + "pdf_ingestion:ingest_pdf", "ingest"),
        (_A + "visual_requalification:requalify_visual_objects", "requalify"),
        (_A + "draft_publication:qualify_draft", "qualify"),
        (_A + "draft_publication:index_draft", "index"),
        (_A + "draft_publication:publish_draft", "publish"),
        (_A + "document_tree_extraction:annotate_document_tree", "tree"),
    ):
        _patch(target, "(阶段其余)", stage=stage)
    # ingest 子步骤
    for target, label in (
        (_A + "pdf_ingestion:_source", "源阶段·打开/落盘(其余)"),
        (
            _A + "pdfspine_document:PdfspineDocumentAdapter.extract_document",
            "pdfspine 抽取·get_text+get_svg",
        ),
        (_A + "aia_processing:ProcessingPipeline.run", "逐页处理·pipeline(其余)"),
        (_A + "page_partition:ModelPagePartitioner.partition", "版面·LLM 路径(其余)"),
        (_A + "deterministic_partition:TextPageRouterPartitioner.partition", "版面·确定性文字页"),
        (
            _A + "deterministic_partition:TextPageRouterPartitioner._tables",
            "版面·find_tables(ruled)",
        ),
        (_A + "onnx_partition:OnnxPagePartitioner.partition", "版面·ONNX(PP-DocLayout)"),
        (_A + "figure_reasoning:render_svg_png", "SVG→PNG(resvg)"),
        (_A + "visual_semantics:_prepare", "VLM 准备·裁图渲染(其余)"),
        (_A + "visual_semantics:VisualSemanticAdapter.infer", "VLM 语义·infer(其余)"),
        (_A + "chart_semantics:ModelChartExtractor.infer", "chart·IR 调用(其余)"),
        (_A + "chart_semantics:ModelDescriptionGenerator.infer", "chart·描述调用(其余)"),
        (_A + "chart_semantics:map_chart", "chart·IR 映射校验"),
        (_A + "semantic_objects:SemanticObjectAdapter._chart", "对象·chart(其余)"),
        (_A + "semantic_objects:SemanticObjectAdapter._table", "对象·table(其余)"),
        (_A + "semantic_objects:SemanticObjectAdapter._table_inferred", "对象·table TSR(其余)"),
        (_A + "semantic_objects:SemanticObjectAdapter._table_rows", "对象·table 按行(其余)"),
        (_A + "semantic_objects:SemanticObjectAdapter._diagram", "对象·diagram(其余)"),
        (_A + "semantic_objects:SemanticObjectAdapter._formula", "对象·formula(其余)"),
        (_A + "semantic_objects:SemanticObjectAdapter._description", "对象·描述(其余)"),
        (_A + "pdfspine_tables:PdfspineTableAdapter.extract", "表格·find_tables"),
        (_A + "pdfspine_tsr:SlanetPlusRecognizer.recognize", "表格·SLANet 推理"),
        (_A + "pdfspine_tsr:infer_table_grid", "表格·TSR 取区域渲染(其余)"),
        (_A + "pdfspine_figure:PdfspineFigureParser.extract", "图·SVG 解析(get_svg+get_text)"),
        (_A + "page_metadata_extraction:annotate_page_metadata", "页元数据(其余)"),
        (_A + "pdf_ingestion:_export_review", "审阅页导出"),
        (_A + "processing_store:ProcessingStore.cache_output", "落盘·store 写"),
        (_A + "processing_store:ProcessingStore.cache", "落盘·store 写"),
        (_A + "processing_store:ProcessingStore.save_draft", "落盘·store 写"),
        (_A + "processing_store:ProcessingStore.publish", "落盘·store 写"),
        (_A + "document_store:LocalDocumentStore.put", "落盘·store 写"),
        (_A + "document_store:LocalDocumentStore.publish", "落盘·store 写"),
        (
            "ragspine.common.evidence.providers.local_models:LocalEmbeddingAdapter.embed_description",
            "embedding",
        ),
        (
            "ragspine.common.evidence.providers.local_models:LocalEmbeddingAdapter.embed_descriptions",
            "embedding",
        ),
        (
            "ragspine.common.evidence.providers.local_models:LocalEmbeddingAdapter.embed_query",
            "embedding",
        ),
    ):
        _patch(target, label)
    _patch(
        _A + "semantic_objects:SemanticObjectAdapter.process", "对象处理(其余)", before=_count_kind
    )
    # 所有 LLM / VLM 调用的唯一入口
    jc = "ragspine.common.evidence.providers.json_completion:JsonCompletionClient."
    _patch(jc + "complete_json", "", llm=True)
    _patch(jc + "complete_text_json", "", llm=True)


print("计时器已定义（尚未安装 patch）")

In [ ]:
# ───────────── 跑：安装计时 → run_folder_pipeline → 恢复 ─────────────
from enterprise_pdf_rag.adapters.folder_pipeline import run_folder_pipeline

_T0 = time.perf_counter()


def progress(event, payload):
    # 只打印文档与阶段的切换，页级进度不打印
    if event in ("discovered", "document_start", "document_done", "done") or (
        event == "document_progress" and "pages_done" not in payload
    ):
        brief = {
            k: v
            for k, v in payload.items()
            if k in ("count", "pdf", "stage", "status", "error", "budget", "pages")
        }
        if "pdf" in brief:
            brief["pdf"] = Path(brief["pdf"]).name
        print(f"[{time.perf_counter() - _T0:7.1f}s]", event, brief)


# 两处只读 patch，让本次只做入库：只测前 MAX_PDFS 份；忽略 .env 里的题集 / NB_REPORT_DIR，不答题、不写 report
_orig_discover = folder_pipeline.discover_pdfs
_orig_get_settings = folder_pipeline.get_settings
folder_pipeline.discover_pdfs = lambda folder: _orig_discover(folder)[:MAX_PDFS]
folder_pipeline.get_settings = lambda: _orig_get_settings().model_copy(
    update={"questions_path": None, "report_dir": None}
)
install_patches()
PATCHED_COUNT = len(_PATCHED)
print(f"已包装 {PATCHED_COUNT} 处绑定；未找到的目标 {len(_SKIPPED)} 个：{_SKIPPED}")
try:
    result = run_folder_pipeline(
        PDF_DIR,
        questions=None,
        ingestion_root=WORK_ROOT,
        max_live_calls_per_pdf=MAX_LIVE_CALLS_PER_PDF,
        max_live_calls_total=MAX_LIVE_CALLS_TOTAL,
        max_parallel_documents=1,
        ingest_mode=INGEST_MODE,
        build_tree=BUILD_TREE,
        unverified_table_structure=EFFECTIVE_TABLE_STRUCTURE,
        layout_policy=EFFECTIVE_LAYOUT,
        requalify=REQUALIFY,
        report_dir=None,
        progress=progress,
    )
finally:
    restore_patches()
    folder_pipeline.discover_pdfs = _orig_discover
    folder_pipeline.get_settings = _orig_get_settings
WALL_S = time.perf_counter() - _T0
print(f"完成：wall {WALL_S:.1f}s | ok={result.ok} | live_calls={result.live_calls.model_dump()}")

In [ ]:
# ───────────── 汇总：打印 [A] [A2] [B] [C]，写 summary.json / summary.md ─────────────
import json
import statistics
from collections import Counter

STAGES = ("ingest", "requalify", "qualify", "index", "publish", "tree")


def fmt_s(value):
    return f"{value:.1f}" if value < 10 else f"{value:.0f}"


def render(rows, columns):
    cells = [[str(row.get(col, "")) for col in columns] for row in rows]
    widths = [max([len(col), *(len(line[i]) for line in cells)]) for i, col in enumerate(columns)]
    out = ["  ".join(col.ljust(w) for col, w in zip(columns, widths))]
    out += ["  ".join(v.ljust(w) for v, w in zip(line, widths)) for line in cells]
    return "\n".join(out)


def render_md(rows, columns):
    out = ["| " + " | ".join(columns) + " |", "|" + "|".join("---" for _ in columns) + "|"]
    out += ["| " + " | ".join(str(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join(out)


docs = [d for d in result.documents if d.status not in ("duplicate_of", "skipped_not_referenced")]

# ---- [A] 阶段耗时：每份 PDF 一行，单元格 = 秒/占总耗时百分比 ----
a_rows, a_json = [], []
for d in docs:
    name = Path(d.pdf_path).name
    total = d.elapsed_s or prof.doc_incl.get(name, 0.0)
    summ = d.ingestion
    kinds = prof.kinds.get(name, {})
    stage_s = {s: prof.stage_incl.get((name, s), 0.0) for s in STAGES}
    other = max(total - sum(stage_s.values()), 0.0)
    row = {
        "pdf": name if len(name) <= 22 else name[:19] + "...",
        "状态": "ok" if d.status == "published" else d.status[:8],
        "页": "" if summ is None else summ.source_page_count,
        "图": kinds.get("CHART", 0),
        "表": kinds.get("TABLE", 0),
        "总s": fmt_s(total),
    }
    for s in STAGES:
        row[s] = f"{fmt_s(stage_s[s])}/{round(100 * stage_s[s] / total) if total else 0}%"
    row["其他"] = f"{fmt_s(other)}/{round(100 * other / total) if total else 0}%"
    a_rows.append(row)
    a_json.append(
        {
            "pdf": name,
            "status": d.status,
            "error": d.error,
            "pages": None if summ is None else summ.source_page_count,
            "objects_by_kind": dict(kinds),
            "total_s": round(total, 3),
            "stage_s": {s: round(v, 3) for s, v in stage_s.items()},
            "other_s": round(other, 3),
            "live_calls": d.live_calls,
            "retries": d.retries,
            "transient_failures": d.transient_failures,
        }
    )
tot_all = sum(j["total_s"] for j in a_json)
if len(a_rows) > 1 and tot_all:
    agg = {s: sum(j["stage_s"][s] for j in a_json) for s in STAGES}
    other_all = max(tot_all - sum(agg.values()), 0.0)
    row = {
        "pdf": "合计",
        "状态": "",
        "页": sum(r["页"] for r in a_rows if r["页"] != ""),
        "图": sum(r["图"] for r in a_rows),
        "表": sum(r["表"] for r in a_rows),
        "总s": fmt_s(tot_all),
    }
    for s in STAGES:
        row[s] = f"{fmt_s(agg[s])}/{round(100 * agg[s] / tot_all)}%"
    row["其他"] = f"{fmt_s(other_all)}/{round(100 * other_all / tot_all)}%"
    a_rows.append(row)
A_COLS = ["pdf", "状态", "页", "图", "表", "总s", *STAGES, "其他"]

# ---- [A2] 子步骤排名（自身耗时，全部 PDF 合计）----
sub = defaultdict(float)
sub_calls = defaultdict(int)
sub_by_pdf = defaultdict(lambda: defaultdict(float))
for (pdf, stage, label), secs in prof.excl.items():
    if pdf is None:  # preflight 的 embedding 探测发生在任何文档之前，不属于某份 PDF
        continue
    sub[(stage, label)] += secs
    sub_calls[(stage, label)] += prof.calls[(pdf, stage, label)]
    sub_by_pdf[pdf][f"{stage}/{label}"] += secs
ranked = sorted(sub.items(), key=lambda kv: -kv[1])
a2_rows = []
for (stage, label), secs in ranked[:14]:
    a2_rows.append(
        {
            "阶段": stage if stage != "-" else "文档",
            "子步骤": label,
            "s": fmt_s(secs),
            "占总%": round(100 * secs / tot_all) if tot_all else 0,
            "次数": sub_calls[(stage, label)],
        }
    )
rest = sum(secs for _, secs in ranked[14:])
if rest > 0:
    a2_rows.append(
        {
            "阶段": "",
            "子步骤": f"其余 {len(ranked) - 14} 项",
            "s": fmt_s(rest),
            "占总%": round(100 * rest / tot_all) if tot_all else 0,
            "次数": "",
        }
    )
A2_COLS = ["阶段", "子步骤", "s", "占总%", "次数"]
coverage = sum(sub.values()) / tot_all if tot_all else 0

# ---- [B] LLM 调用按 task ----
by_task = defaultdict(list)
for rec in prof.llm:
    by_task[rec["task"]].append(rec)
b_rows, b_json = [], []


def llm_row(task, recs):
    secs = [r["ms"] / 1000 for r in recs]
    tin = [r["in_tokens"] for r in recs if r["in_tokens"] is not None]
    tout = [r["out_tokens"] for r in recs if r["out_tokens"] is not None]
    return {
        "task": task,
        "次": len(recs),
        "命中": sum(r["cache_hit"] for r in recs),
        "失败": sum(bool(r["error"]) for r in recs),
        "总s": fmt_s(sum(secs)),
        "均s": f"{statistics.fmean(secs):.1f}",
        "最大s": f"{max(secs):.1f}",
        "入tok": sum(tin) if tin else "-",
        "出tok": sum(tout) if tout else "-",
    }, sum(secs)


for task, recs in sorted(by_task.items(), key=lambda kv: -sum(r["ms"] for r in kv[1])):
    row, _ = llm_row(task, recs)
    b_rows.append(row)
    b_json.append({**row, "by_pdf": dict(Counter(r["pdf"] for r in recs))})
if prof.llm:
    b_rows.append(llm_row("合计", prof.llm)[0])
else:
    b_rows.append({"task": "(本次没有任何 LLM 调用)"})
B_COLS = ["task", "次", "命中", "失败", "总s", "均s", "最大s", "入tok", "出tok"]

# ---- [C] 磁盘产物：WORK_ROOT/<sha256>/ 下全部文件 ----
c_rows, c_json = [], []
for d in docs:
    name = Path(d.pdf_path).name
    root = WORK_ROOT / (d.sha256 or "")
    by_ext = defaultdict(lambda: [0, 0])
    by_dir = defaultdict(lambda: [0, 0])
    if d.sha256 and root.is_dir():
        for path in root.rglob("*"):
            if path.is_file() and not path.is_symlink():
                size = path.stat().st_size
                ext = path.suffix.lower() or "(无扩展名)"
                by_ext[ext][0] += 1
                by_ext[ext][1] += size
                top = path.relative_to(root).parts[0]
                sub_dir = (
                    "/".join(path.relative_to(root).parts[:2])
                    if top == "processing" and len(path.relative_to(root).parts) > 2
                    else top
                )
                by_dir[sub_dir][0] += 1
                by_dir[sub_dir][1] += size
    files = sum(v[0] for v in by_ext.values())
    nbytes = sum(v[1] for v in by_ext.values())
    top_ext = "; ".join(
        f"{e} {n}/{b / 1e6:.1f}M"
        for e, (n, b) in sorted(by_ext.items(), key=lambda kv: -kv[1][1])[:4]
    )
    c_rows.append(
        {
            "pdf": name if len(name) <= 22 else name[:19] + "...",
            "文件": files,
            "MB": f"{nbytes / 1e6:.1f}",
            "后端": d.object_backend or "-",
            "扩展名 数量/大小(前4)": top_ext,
        }
    )
    c_json.append(
        {
            "pdf": name,
            "files": files,
            "bytes": nbytes,
            "object_backend": d.object_backend,
            "by_ext": {e: {"files": n, "bytes": b} for e, (n, b) in sorted(by_ext.items())},
            "by_dir": {e: {"files": n, "bytes": b} for e, (n, b) in sorted(by_dir.items())},
        }
    )
C_COLS = ["pdf", "文件", "MB", "后端", "扩展名 数量/大小(前4)"]

# ---- 写文件 ----
OUT_DIR.mkdir(parents=True, exist_ok=True)
meta = {
    "pdf_dir": str(PDF_DIR),
    "selected_pdfs": [p.name for p in SELECTED_PDFS],
    "cold_start": COLD_START,
    "ingest_mode": INGEST_MODE,
    "layout": EFFECTIVE_LAYOUT,
    "table_structure": EFFECTIVE_TABLE_STRUCTURE,
    "requalify": REQUALIFY,
    "build_tree": BUILD_TREE,
    "wall_s": round(WALL_S, 3),
    "sum_documents_s": round(tot_all, 3),
    "span_coverage": round(coverage, 4),
    "live_calls": result.live_calls.model_dump(),
    "ingestion_root": str(WORK_ROOT),
    "patched_bindings": PATCHED_COUNT,
    "targets_not_found": _SKIPPED,
}
summary = {
    "meta": meta,
    "A_stages": a_json,
    "A2_substeps": [
        {"stage": s, "label": lab, "seconds": round(v, 3), "calls": sub_calls[(s, lab)]}
        for (s, lab), v in ranked
    ],
    "A2_by_pdf": {p: {k: round(v, 3) for k, v in d_.items()} for p, d_ in sub_by_pdf.items()},
    "B_llm": b_json,
    "B_llm_calls": prof.llm,
    "C_artifacts": c_json,
}
(OUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8"
)

md_text = (
    "\n\n".join(
        [
            "# 计时测量 summary",
            f"- 冷启动: {COLD_START} | 模式: {INGEST_MODE} / 版面 {EFFECTIVE_LAYOUT} / 表格 {EFFECTIVE_TABLE_STRUCTURE} | 文档合计 {fmt_s(tot_all)}s，wall {fmt_s(WALL_S)}s，子步骤覆盖 {round(100 * coverage)}%",
            "## [A] 阶段耗时（单元格 = 秒/占该 PDF 总耗时%）\n\n" + render_md(a_rows, A_COLS),
            "## [A2] 子步骤耗时排名（自身耗时，扣除被包子调用）\n\n" + render_md(a2_rows, A2_COLS),
            "## [B] LLM 调用（按 task）\n\n" + render_md(b_rows, B_COLS),
            "## [C] 磁盘产物\n\n" + render_md(c_rows, C_COLS),
        ]
    )
    + "\n"
)
(OUT_DIR / "summary.md").write_text(md_text, encoding="utf-8")

# ---- 打印 ----
print(
    f"冷启动={COLD_START} | {INGEST_MODE}/{EFFECTIVE_LAYOUT}/{EFFECTIVE_TABLE_STRUCTURE} | 文档合计 {fmt_s(tot_all)}s wall {fmt_s(WALL_S)}s | 子步骤覆盖 {round(100 * coverage)}%"
)
print("\n[A] 阶段耗时（格式 秒/占比）")
print(render(a_rows, A_COLS))
print("\n[A2] 子步骤耗时排名（自身耗时）")
print(render(a2_rows, A2_COLS))
print("\n[B] LLM 调用（按 task）")
print(render(b_rows, B_COLS))
print("\n[C] 磁盘产物")
print(render(c_rows, C_COLS))
for d in docs:
    if d.status != "published" or d.error:
        print(f"[!] {Path(d.pdf_path).name}: {d.status} {d.error or ''}"[:200])
print("\n已写出:", OUT_DIR / "summary.json")
print("已写出:", OUT_DIR / "summary.md")